In [ ]:
from transformers import BigBirdTokenizer
import pandas as pd
import os
from tqdm import tqdm
import json

In [ ]:
model_id = 'google/bigbird-roberta-base'
tokenizer = BigBirdTokenizer.from_pretrained(model_id, trust_remote_code=True)

## QA Dataset

* all conflicting datapoint questions with their best paraphrase

In [ ]:
with open('raw_conflicting/train_conflict.json', 'r') as f:
     train_conflict = json.load(f)

In [ ]:
conflict_questions = {conflict['question'] for conflict in train_conflict}

with open('paraphrases/train-paraphrases.jsonl', 'r') as f:
    linesp = f.read().strip().split('\n')
    train_with_paraphrases = [json.loads(line) for line in linesp]
    train_paraphrases_clean = []
    for train_para in train_with_paraphrases:
        if train_para['question'] in conflict_questions:
            if 'question_op_bertscore_f1' not in train_para:
                paraphrased = ""
            else:
                max_idx = train_para['question_op_bertscore_f1'].index(max(train_para['question_op_bertscore_f1']))
                paraphrased = train_para['question_op'][max_idx]
            train_paraphrases_clean.append({
                'question': train_para['question'],
                'paraphrase': paraphrased
            })

with open('raw_baseline_data/train.jsonl', 'r') as f:
    lines = f.read().strip().split('\n')
    train_clean = [json.loads(line) for line in lines]
    test_data = []
    for test_example in tqdm(train_clean, desc='converting to eval QA dataset'):
        if test_example['question'] in conflict_questions:
            paraphrase = None
            for paraphrased in train_paraphrases_clean:
                if test_example['question'] == paraphrased['question']:
                    paraphrase = paraphrased['paraphrase']
                    break

            test_dict = {
                'question': test_example['question_upd'],
                'q-paraphrased': paraphrase if paraphrase is not None else test_example['question_upd'],
                'subject_name': test_example['subject_name'],
                'cop': test_example['cop'],
                'opa': test_example['opa_upd'],
                'opb': test_example['opb_upd'],
                'opc': test_example['opc_upd'],
                'opd': test_example['opd_upd']
            }

            test_data.append(test_dict)

print(len(test_data))
with open('final_data/qa/test.json', 'w') as fw:
    json.dump(test_data, fw, indent=2)

## Domain Adaptation Data - Baseline

* all conflicting data points clean contexts

In [ ]:
## should give 14_540
print(len(conflict_questions))

In [ ]:
def convert_to_csv(input_path, output_path, tokenizer):
    assert os.path.exists(input_path)

    if not os.path.exists(output_path):
        os.makedirs(output_path)

    all_tokens_da = 0
    for train_or_dev in tqdm(os.listdir(input_path), 'Conversion to csv...'):
        train_or_dev_path = os.path.join(input_path, train_or_dev)
        df = {'entries': []}

        with open(train_or_dev_path, 'r') as f:
            objects = f.read().strip().split('\n')
            entries = [json.loads(obj) for obj in objects]

        ## train
        if train_or_dev == 'train.jsonl':

            raw_corpus = ""
            limit, allow_more_clean = 0, 15_000
            for entry in entries:
                if entry['question'] not in conflict_questions:
                    if limit < allow_more_clean:
                        limit += 1
                    else:
                        continue
                if entry['exp_to_edit'] is None:
                    continue
                else:
                    if entry['exp_to_edit']:
                        if 'exp_upd' in entry and entry['exp_upd'] is not None:
                            raw_corpus += entry['exp_upd'] + '\n\n'
                            df['entries'].append(entry['exp_upd'])
                    else:
                        raw_corpus += entry['exp'] + '\n\n'
                        df['entries'].append(entry['exp'])
        ## dev
        else:
            raw_corpus = ""
            for entry in entries:
                if entry['exp_to_edit'] is None:
                    continue
                else:
                    if entry['exp_to_edit']:
                        if 'exp_upd' in entry and entry['exp_upd'] is not None:
                            raw_corpus += entry['exp_upd'] + '\n\n'
                            df['entries'].append(entry['exp_upd'])
                    else:
                        raw_corpus += entry['exp'] + '\n\n'
                        df['entries'].append(entry['exp'])

        raw_corpus = raw_corpus.strip()
        tokens_corpus = tokenizer.tokenize(raw_corpus)

        print(
            f'\n\n{train_or_dev} has {round(len(tokens_corpus) / (10 ** 6), 4)}M tokens for domain adaptation\n\n')

        all_tokens_da += len(tokens_corpus)

        df = pd.DataFrame(df)

        savename = 'dev.csv' if 'dev' in train_or_dev else 'train.csv'

        df.to_csv(os.path.join(output_path, savename), index=False)

    print(f'Total of {round(all_tokens_da / (10 ** 9), 4)}B tokens for domain adaptation')


convert_to_csv('raw_baseline_data', 'final_data/baseline_corpus', tokenizer)

**Train Tokens before upsampling = 2.147M**

**Train Tokens after upsampling = 3.346M**

**Dev Tokens = 0.2603M**


## Domain Adaptation Data - Conflicts

* all conflicting data points modified + clean contexts

In [ ]:
def convert_to_csv(input_path, output_path, tokenizer):
    assert os.path.exists(input_path)

    if not os.path.exists(output_path):
        os.makedirs(output_path)

    all_tokens_da = 0
    for train_or_dev in tqdm(os.listdir(input_path), 'Conversion to csv...'):
        if train_or_dev != 'train_conflict.json':
            continue
        train_or_dev_path = os.path.join(input_path, train_or_dev)

        df = {'entries': []}

        with open('raw_baseline_data/train.jsonl', 'r') as f:
            lines = f.read().strip().split('\n')
            clean_data = [json.loads(l) for l in lines]

        unique_questions = set()
        with open(train_or_dev_path, 'r') as f:
            entries = json.load(f)

            ## take all conflicts
            raw_corpus = ""
            for entry in entries:
                unique_questions.add(entry['question'])
                raw_corpus += entry['mod_context'] + '\n\n'
                df['entries'].append(entry['mod_context'])

            ## sample the cleaned contexts from the baseline data
            ## which do appear in the conflicting data
            for entry in clean_data:
                if entry['question'] in unique_questions:
                    if entry['exp_to_edit'] is None:
                        continue
                    else:
                        if entry['exp_to_edit']:
                            if 'exp_upd' in entry and entry['exp_upd'] is not None:
                                raw_corpus += entry['exp_upd'] + '\n\n'
                                df['entries'].append(entry['exp_upd'])
                        else:
                            raw_corpus += entry['exp'] + '\n\n'
                            df['entries'].append(entry['exp'])

            raw_corpus = raw_corpus.strip()
            tokens_corpus = tokenizer.tokenize(raw_corpus)

            print(
                f'\n\n{train_or_dev} has {round(len(tokens_corpus) / (10 ** 6), 4)}M tokens for domain adaptation\n\n')

            all_tokens_da += len(tokens_corpus)

        df = pd.DataFrame(df)

        savename = 'dev.csv' if 'dev' in train_or_dev else 'train.csv'

        df.to_csv(os.path.join(output_path, savename), index=False)

    print(f'Total of {round(all_tokens_da / (10 ** 9), 4)}B tokens for domain adaptation')


convert_to_csv('raw_conflicting_data', 'final_data/conflict_corpus', tokenizer)

**Train Tokens Conflicting Data = 3.4315M**